### TT06 Filter Sobel by Diana Natali Maldonado

gray_scale_core, sobel_core/sobel_control (the algorithm), spi_core/spi_control (the host interface), tt_um_gray_sobel (the Tiny Tapeout wrapper). 


Algorith                        host interface       tiny tapeout wrapper
- gray scale core               - spi core           - tt um gray sobel
- sobel core/ sobel control     - spi control

It's a Tiny Tapeout tile (1×2, 10 MHz) that processes pixels streamed over SPI. Pipeline (top_gray_sobel.sv):

RGB pixel (24-bit) ──SPI──▶ gray_scale_core (RGB→gray 8-bit)   ─▶ sobel_control/sobel_core (edge magnitude)─▶  result ──SPI ──▶ out
                                  
                              
Key parts:
- spi_control + spi_core = an SPI slave. The host clocks in a 24-bit RGB pixel (spi_sdi_i) and clocks out the processed pixel (spi_sdo_o). (spi_control.sv:71 — WORD_SIZE = MAX_PIXEL_BITS, moved as 3 bytes.)
- *** select_process_i[1:0] *** (pins ui[3:4]) picks the mode (top_gray_sobel.sv:55):

- *00* full pipeline (gray→sobel),
  - *01* sobel only,
  - *10* grayscale only,
  - *11* bypass
    
- start_sobel_i (ui[5]) kicks off Sobel.
- LFSR (pins uio[0:2]) is just an on-chip test-pattern generator so you can exercise it without a real image.
- Handshake is px_rdy (valid) signals — classic streaming/ready protocol.

So Diana's block is a "feed a pixel, get a filtered pixel back" coprocessor, with SPI as its front door.

Your femto SoC — SOC_flash.v (memory-mapped RISC-V host)

A FemtoRV32 CPU with a simple address-decoded bus (SOC_flash.v:131):


| Addr ([31:16]) |Peripheral  |
| ---            | ---        |
| 0x0000         | RAM (bram) |
| 0x0040         | UART       |
| 0x0041         | GPIO       |
| 0x0042         | mult       |
| 0x0043         | div        |
| 0x0044         | bin2bcd    |


Plus MappedSPIFlash — but note that SPI is dedicated to instruction fetch from flash, not a general peripheral bus.

How they connect

Diana already split the algorithm (top_gray_sobel) from its SPI wrapper — that's the gift. Two ways to wire it to femto:

Option A — On-chip, memory-mapped (recommended for the ASIC/embedded SoC):
Add a new peripheral peripheral_sobel at a free chip-select (e.g. 0x0045) that instantiates top_gray_sobel directly (skip the SPI). The CPU then:
*(SOBEL_DATA)  = rgb_pixel;   // write → in_pixel_i + px_rdy_i pulse
*(SOBEL_CTRL)  = mode|start;  // select_process_i / start_sobel_i
result = *(SOBEL_DATA);       // read out_pixel_o when px_rdy_o
This fits your existing cs/MUX decoder pattern perfectly and is the tightest integration for a single Tiny Tapeout die.

Option B — Keep the SPI (two-chip / faithful-to-TT):
Add an SPI master peripheral to femto that talks to Diana's unmodified SPI slave + a couple of GPIO lines for select/start. Use this if the Sobel stays a separate chip (e.g., her actual TT06 silicon) and femto is the controller.

The full embedded picture (with your iCESugar i9+ PMODs)

![photo1](img/pho.png)

 
Same RTL targets both your goals: build it on the iCESugar i9+ (FPGA prototype, real image in/out via the PMODs) and push it through OpenLane→sky130 for the Tiny Tapeout ASIC.



First, here's the course map — your project, taught from the ground up:

┌───────┬──────────────────────────────────────────────────────────────────┐
│ Class │                              Topic                               │
├───────┼──────────────────────────────────────────────────────────────────┤
│ 0     │ 🗺️ The big picture (this one) — what we're building & the layers │
├───────┼──────────────────────────────────────────────────────────────────┤
│ 1     │ Digital logic refresher — bits, gates, registers, the clock      │
├───────┼──────────────────────────────────────────────────────────────────┤
│ 2     │ What a CPU is — RISC-V & your FemtoRV32, the memory bus          │
├───────┼──────────────────────────────────────────────────────────────────┤
│ 3     │ The Sobel algorithm — grayscale + edge detection → hardware      │
├───────┼──────────────────────────────────────────────────────────────────┤
│ 4     │ Connecting CPU ↔ filter — peripherals, SPI, data flow            │
├───────┼──────────────────────────────────────────────────────────────────┤
│ 5     │ Making it real — FPGA (iCESugar) vs ASIC (sky130/Tiny Tapeout)   │
├───────┼──────────────────────────────────────────────────────────────────┤
│ 6     │ Verification & bring-up — simulate, test, demo                   │
└───────┴──────────────────────────────────────────────────────────────────┘

---
📘 Class 0 — The Big Picture

What are we building?

A tiny camera-brain on a chip: it takes an image, and finds the edges in it (outlines of objects). That's the Sobel filter. To make it useful, we wrap it with a small computer (a RISC-V microcontroller) that loads the image, feeds it to the filter, and shows the result on a screen.

▎ One sentence: a small computer + an edge-detection accelerator, built as a real chip.

The layers (this is the key idea)

Hardware is built in layers of abstraction — like a building. From top (closest to you) to bottom (closest to silicon):

![pho 2](img/pho2.png)

Each layer hides the one below it. You write at the top two layers (software + RTL); the tools you installed automatically translate down to the bottom:
- Yosys turns RTL → gates (synthesis)
- OpenLane turns gates → silicon layout (sky130 ASIC), or nextpnr maps gates → FPGA
- Xyce / ngspice check the analog/transistor behavior

Two destinations, same design

Your RTL can go two ways — and you can do both:
- 🔵 FPGA (your iCESugar i9+): reprogrammable, fast to test, real image in/out via your PMODs. Prototype.
- 🟢 ASIC (sky130 via Tiny Tapeout): a permanent physical chip. The final product.

The vocabulary you'll hear from me

- RTL = "Register-Transfer Level" — Verilog/SystemVerilog code describing the circuit
- Synthesis = RTL → logic gates
- Netlist = the list of gates and their wiring
- GDSII = the final chip-layout file (what gets manufactured)
- PDK = Process Design Kit (sky130) — the "rules + parts" for a specific factory
- Peripheral = a hardware block the CPU controls via memory addresses



📘 Class 1 — Digital Logic (the foundation)

Everything in your chip is built from just two kinds of circuits: ones that react and ones that remember. Master that distinction and the rest follows. Let's build up.

---
1. Bits — the alphabet

Hardware only knows two values: 0 (low voltage) and 1 (high voltage). A single wire carries one bit. That's it. Everything — pixels, instructions, addresses — is just bundles of bits.

A bundle of wires is a bus. In Verilog you'll see:
wire [7:0] pixel;   // 8 wires = 8 bits = one byte (values 0..255)
Your grayscale pixel is 8 bits (0=black … 255=white). Diana's RGB pixel is 24 bits (8 red + 8 green + 8 blue).

---
2. Combinational logic — circuits that react

These compute an output purely from the current inputs — no memory. Built from gates:

┌──────┬─────────────────────────┬─────────────┐
│ Gate │         Meaning         │   Example   │
├──────┼─────────────────────────┼─────────────┤
│ AND  │ 1 only if both inputs 1 │ 1 AND 0 = 0 │
├──────┼─────────────────────────┼─────────────┤
│ OR   │ 1 if either input 1     │ 1 OR 0 = 1  │
├──────┼─────────────────────────┼─────────────┤
│ NOT  │ flips the bit           │ NOT 0 = 1   │
├──────┼─────────────────────────┼─────────────┤
│ XOR  │ 1 if inputs differ      │ 1 XOR 1 = 0 │
└──────┴─────────────────────────┴─────────────┘

The output changes the instant an input changes (after a tiny delay). In Verilog, combinational logic is written with assign or always_comb:
assign in_px_sobel = select_sobel_mux ? in_pixel_i[7:0] : out_px_gray;

👆 That's a real line from Diana's top_gray_sobel.sv — a multiplexer (a "switch"): pick one of two inputs based on a select bit. Pure combinational.

3. Sequential logic — circuits that remember

To build anything useful (counters, state machines, pipelines) hardware needs memory. The atom of memory is the flip-flop (a 1-bit storage cell). A register is just several flip-flops side by side (e.g. an 8-bit register stores a byte).

A flip-flop has a magic rule:

▎ It only updates its stored value on the rising edge of the clock — and holds it steady the rest of the time.

In Verilog, that's always_ff @(posedge clk):
always_ff @(posedge clk_i or negedge nreset_i) begin
    if(!nreset_i)        data_tx <= '0;        // reset: clear
    else if(px_rdy_o_spi_i) data_tx <= output_px_sobel_i;  // capture on clock edge
end
👆 Also straight from Diana's spi_control.sv. data_tx is a register that captures and holds a pixel.

▎ Mental model: sequential = a whiteboard that's only erasable/writable at each clock tick.

---
4. The clock & reset — the heartbeat

- Clock (clk): a signal going 0→1→0→1… millions of times/sec (your chip: 10 MHz = 10 million ticks/sec). Every register updates in lockstep on the rising edge. This is synchronous design — the discipline that makes big chips actually work.
- Reset (rst_n / nreset_i): a signal to force all registers to a known starting state at power-on. The n means active-low (reset happens when it's 0). That's why you see if(!nreset_i).

---

5. The ready/valid handshake — how blocks talk

Here's the concept that unlocks Diana's whole design. When one block sends data to another, how does the receiver know when the data is valid? A handshake:
![pho3](img/pho3.png)
In her code, px_rdy_i / px_rdy_o are exactly these "valid" pulses. The gray-scale block raises px_rdy_o for one clock when its output pixel is ready; the Sobel block watches that to know when to grab it. This is how pixels stream through the pipeline, one valid beat at a time.


---
🧠 Recap

1. Bits on wires; bundles = buses ([7:0]).
2. Combinational (assign) = reacts instantly, no memory.
3. Sequential (always_ff @(posedge clk)) = remembers, updates only on the clock edge.
4. Clock = the heartbeat; reset = known starting point (active-low here).
5. ready/valid (px_rdy) = the handshake that streams data between blocks.

Quick check (answer in your head 🙂)

- Is assign uo_out[0] = spi_sdo_o; combinational or sequential?
- Why must a flip-flop wait for the clock edge instead of updating immediately?

(Answers: combinational — it's a direct wire connection, no clock. And: so all parts of the chip change together in a predictable, race-free way.)


📘 Class 2 — What a CPU is (your FemtoRV32)

In Class 1 you met the two atoms of hardware: **combinational** logic (reacts) and **sequential** logic (remembers). A **CPU is just those two atoms, arranged into a loop that reads instructions and obeys them.** Nothing more magic than that.

Your project uses **FemtoRV32** — a family of tiny RISC-V cores by Bruno Levy. You have ten variants in `cores/cpu/` (quark, electron, gracilis, tachyon…); the one wired into your chip is the **Quark** (`femtorv32_quark.v`) — the smallest, ~200 lines, implementing the base **RV32I** instruction set.

▎ One sentence: the CPU is the "small computer" from Class 0 — it loads the image, feeds pixels to the Sobel filter, and reads the result back.

What "RISC-V" means
- **RISC-V** = an open, free instruction set (the "language" the CPU speaks). No license fees — perfect for a thesis / Tiny Tapeout.
- **RV32** = 32-bit registers and addresses.
- **I** = the base *Integer* instruction set: ~40 instructions (add, sub, load, store, branch, jump…). That's all Quark needs.


1. The heartbeat of a CPU — fetch / decode / execute

Every CPU, from your Quark to the chip in your laptop, runs the same eternal loop:

```
┌─────────┐   ┌──────────┐   ┌──────────┐
│  FETCH  │──▶│  DECODE  │──▶│ EXECUTE  │──┐
│ get next│   │ what is  │   │ do it +  │  │
│ instr   │   │ this?    │   │ store    │  │
└─────────┘   └──────────┘   └──────────┘  │
     ▲                                     │
     └─────────────────────────────────────┘
```

In the Quark this loop is a tiny **state machine** (remember Class 1 — sequential logic). Its states (`femtorv32_quark.v:295`):

```verilog
localparam FETCH_INSTR_bit     = 0;  // put PC on the address bus, ask memory
localparam WAIT_INSTR_bit      = 1;  // wait for memory to return the instruction
localparam EXECUTE_bit         = 2;  // decode + do the work, compute next PC
localparam WAIT_ALU_OR_MEM_bit = 3;  // wait if a load/store/shift is still busy
```

👆 Four states, cycling forever. FETCH and EXECUTE are split by WAIT states for exactly the ready/valid reason from Class 1: memory isn't instant, so the CPU waits for it (`mem_rstrb` asks, `mem_rbusy` says "not yet").

The PC (Program Counter)

The **PC** is a register holding the address of the current instruction. After each one it advances — usually +4 (instructions are 4 bytes), or jumps elsewhere for a branch/jump (`femtorv32_quark.v:354`):

```verilog
PC <= isJALR            ? {aluPlus[ADDR_WIDTH-1:1],1'b0} : // jump to computed addr
      isBranch & taken  ? PCplusImm :                      // take the branch
                          PCplus4;                         // else: next instruction
```

▎ Mental model: the PC is the CPU's finger tracking "which line am I on".


2. Decoding — turning 32 bits into an action

A fetched instruction is just a 32-bit number. **Decoding** is pure combinational logic that slices those bits to figure out *what* to do and *on what*. The Quark reads the opcode field `instr[6:2]` (`femtorv32_quark.v:80`):

```verilog
wire isLoad   = (instr[6:2] == 5'b00000); // rd <- mem[rs1+imm]
wire isStore  = (instr[6:2] == 5'b01000); // mem[rs1+imm] <- rs2
wire isALUreg = (instr[6:2] == 5'b01100); // rd <- rs1 OP rs2
wire isBranch = (instr[6:2] == 5'b11000); // if(rs1 OP rs2) jump
```

Each `isXxx` is one wire that's `1` when the instruction is that type. They steer everything downstream — no clock involved, so this is **combinational** (Class 1).

3. The register file — the CPU's scratchpad

RV32 has **32 general-purpose registers**, each 32 bits — the fast working memory right next to the ALU. In Verilog that's literally an array (`femtorv32_quark.v:100`):

```verilog
reg [31:0] registerFile [31:0];   // 32 registers x 32 bits

always @(posedge clk) begin
   if (writeBack)                          // when an instruction produces a result...
     registerFile[rdId] <= writeBackData;  // ...store it in register rdId
end
```

👆 `rdId = instr[11:7]` — the instruction itself names which register receives the result. This write happens only on the clock edge → **sequential** logic, exactly Class 1.

The **ALU** (Arithmetic Logic Unit) is the combinational block that actually computes add / sub / and / or / shift / compare (`femtorv32_quark.v:142`, the big `aluOut = …`). Inputs: two registers (or a register + an immediate constant). Output: the value written back.


4. The memory bus — how the CPU touches the world

The CPU core is sealed; it talks to *everything* through **one bus** — a handful of wires (`femtorv32_quark.v:38`):

```
┌───────────┬─────┬──────────────────────────────────────┐
│   Wire    │ Dir │              Meaning                 │
├───────────┼─────┼──────────────────────────────────────┤
│ mem_addr  │ out │ which address I want                 │
│ mem_wdata │ out │ data I'm writing                     │
│ mem_rdata │ in  │ data coming back to me               │
│ mem_wmask │ out │ which of the 4 bytes to write        │
│ mem_rstrb │ out │ "read now" strobe                    │
│ mem_rbusy │ in  │ "memory still busy — wait"           │
└───────────┴─────┴──────────────────────────────────────┘
```

This is the same **ready/valid handshake** from Class 1, one level up: the CPU raises `mem_rstrb` to read and waits while `mem_rbusy` is high.

▎ Key idea: to the CPU, **everything is a memory address** — RAM, the UART, the multiplier, and (soon) the Sobel filter all look like memory. This is **memory-mapped I/O**, and it's the bridge to your accelerator.


5. Your SoC — one CPU, many peripherals (`SOC_flash.v`)

`SOC_flash.v` is the **System-on-Chip**: it instantiates the FemtoRV32 and wires its bus to several peripherals. A small **address decoder** looks at the *top* half of the address (`mem_address[31:16]`) and raises one **chip-select** line (`SOC_flash.v:131`):

```verilog
case (mem_address[31:16])     // upper 16 bits pick the peripheral
  16'h0000: cs = 7'b0000001;  // RAM   (program memory)
  16'h0040: cs = 7'b0100000;  // UART  (serial console)
  16'h0041: cs = 7'b0010000;  // GPIO
  16'h0042: cs = 7'b0001000;  // mult  (your hardware multiplier)
  16'h0043: cs = 7'b0000100;  // div
  16'h0044: cs = 7'b0000010;  // bin2bcd
  16'h0001: cs = 7'b1000000;  // dpRAM
endcase
```

Each peripheral listens only when its `cs` bit is high — e.g. the multiplier: `peripheral_mult mult1 ( … .cs(cs[3]) … )` (`SOC_flash.v:100`). On a read, a matching mux drives that peripheral's data onto `mem_rdata` (`SOC_flash.v:145`).

So a C line `*(int*)0x00420004 = 7;` becomes: address `0x0042…` → decoder raises `cs[3]` → the multiplier captures the value. **That is how software drives hardware.**

🔗 Connecting to Class 0

Notice the table stops at `0x0044`. The next free slot, **`0x0045`**, is exactly where **Option A** from the integration analysis (cell 1) adds the Sobel peripheral. Class 4 will wire it in — you now understand the bus it plugs into.


---
🧠 Recap

1. A CPU = the Class-1 atoms arranged as a **fetch → decode → execute** loop.
2. FemtoRV32 **Quark** runs **RV32I**; its loop is a **4-state machine** (FETCH / WAIT_INSTR / EXECUTE / WAIT_ALU_OR_MEM).
3. The **PC** points at the current instruction; **decode** slices `instr` bits into `isLoad / isStore / …`; the **register file** (32×32) + **ALU** do and store the work.
4. The CPU reaches the world through **one memory bus** (addr / wdata / rdata + a ready/valid handshake).
5. **Memory-mapped I/O**: an address decoder turns the top address bits into **chip-selects** — every peripheral looks like memory. Free slot `0x0045` awaits the Sobel block.

Quick check (answer in your head 🙂)

- If software writes to address `0x00410000`, which peripheral responds — and which `cs` bit?
- Why does the CPU need the `mem_rbusy` input — what would break without it?
- Is the instruction decoder (`isLoad = instr[6:2]==…`) combinational or sequential?

*(Answers: GPIO — `cs[4]`, from `16'h0041`. Without `mem_rbusy` the CPU would read garbage before slow memory/IO is ready — it would race ahead. Combinational — pure wiring from `instr` bits, no clock.)*

---
*Next up — **Class 3: the Sobel algorithm**, from the math (grayscale + gradient) down to the `gray_scale_core` / `sobel_core` hardware.*


📘 Class 3 — The Sobel algorithm (from math to hardware)

Class 2 gave you the CPU — the *boss*. Class 3 is the *worker* it commands: Diana's edge-detection accelerator. This is the heart of your chip's job.

What is an "edge"?

Look at any photo. An **edge** is a place where brightness changes sharply — the outline of a flower against the sky, the border of an object. Mathematically, a sharp change = a large **gradient** (steep slope in brightness). Edge detection = "find where the image changes fast."

The pipeline (two steps)

```
  RGB pixel (24-bit)          gray pixel (8-bit)        edge strength (8-bit)
 ┌──────────────────┐  step 1 ┌──────────────────┐ step 2 ┌──────────────────┐
 │  gray_scale_core │ ──────▶ │  (3×3 window)    │ ─────▶ │    sobel_core    │
 │  RGB → brightness│         │  sobel_control   │        │  gradient → edge │
 └──────────────────┘         └──────────────────┘        └──────────────────┘
```

- **Step 1 — Grayscale:** collapse the 3 colours into one brightness number. Edges are about *intensity*, not colour, so we throw colour away first.
- **Step 2 — Sobel:** measure how fast brightness changes around each pixel. Big change → bright output (an edge). Flat area → dark output.

▎ One sentence: grayscale answers "how bright is this pixel?", Sobel answers "how different is it from its neighbours?"


Step 1 — Grayscale (`gray_scale_core.sv`)

The eye isn't equally sensitive to red, green, blue. The standard brightness (luma) formula is:

```
Y = 0.299·R + 0.587·G + 0.114·B      (green matters most, blue least)
```

But multiplying by 0.299 needs a **multiplier** — expensive in silicon. Diana's trick: approximate those constants with **shifts and adds only** (remember Class 1 — `>>` is just rewiring, almost free). From `gray_scale_core.sv:29`:

```systemverilog
out_px_gray_o <= (red>>2)+(red>>5) + (green>>1)+(green>>4) + (blue>>4)+(blue>>5);
```

Decode it — `x>>n` means "divide by 2ⁿ":

```
red  : 1/4 + 1/32  = 0.281   ≈ 0.299   ✅
green: 1/2 + 1/16  = 0.562   ≈ 0.587   ✅
blue : 1/16 + 1/32 = 0.094   ≈ 0.114   ✅
```

👆 A multiplier-free luma approximation — *the* classic "hardware-friendly math" move. The 24-bit input is just R, G, B stacked, split by pure wiring (`gray_scale_core.sv:33`):

```systemverilog
assign red   = in_px_rgb_i[23:16];   // top 8 bits
assign green = in_px_rgb_i[15:8];    // middle 8
assign blue  = in_px_rgb_i[7:0];     // bottom 8
```

And `px_rdy_o <= px_rdy_i;` (`:28`) passes the valid pulse along — the Class 1 handshake, one stage down the pipe.


Step 2 — The Sobel operator: two little 3×3 kernels

To measure change *around* a pixel you look at its 8 neighbours — a **3×3 window**. Sobel slides two small matrices (**kernels**) over that window: one feels horizontal change (Gx), one vertical (Gy).

```
        Gx  (left vs right)            Gy  (top vs bottom)
      ┌──────────────┐               ┌──────────────┐
      │ -1   0   +1  │               │ -1  -2  -1   │
      │ -2   0   +2  │               │  0   0   0   │
      │ -1   0   +1  │               │ +1  +2  +1   │
      └──────────────┘               └──────────────┘
```

"Convolving" = multiply each kernel cell by the pixel under it and sum. But the coefficients are only **0, ±1, ±2** — so again **no multipliers**: ±1 is just add/subtract, ×2 is `<<1`. From `sobel_core.sv:20`:

```systemverilog
// Gx: (right column − left column), middle row weighted ×2
assign x_grad = ( (v0.pix2 - v0.pix0)            // top:    +1·right −1·left
              + ( (v1.pix2 - v1.pix0) << 1 )     // middle: +2·right −2·left
              +   (v2.pix2 - v2.pix0) );          // bottom: +1·right −1·left

// Gy: (bottom row − top row), middle column weighted ×2
assign y_grad = ( (v2.pix0 - v0.pix0)
              + ( (v2.pix1 - v0.pix1) << 1 )
              +   (v2.pix2 - v0.pix2) );
```

👆 `v0/v1/v2` are the three rows of the window — the `sobel_matrix` struct from `parameters.svh:13` (a packed 3×3 of signed 8-bit pixels). The columns `0` (left) cancel against `2` (right) in Gx, exactly the kernel above. **Pure combinational** — the gradients appear the instant the window is filled.


Step 2 (cont.) — Magnitude and saturation

`x_grad`/`y_grad` can be negative (brightness can rise *or* fall). The edge **strength** is their combined size. True magnitude is √(Gx²+Gy²) — but a square root is costly, so Sobel uses the cheap approximation **|Gx| + |Gy|** (`sobel_core.sv:29`):

```systemverilog
assign abs_x_grad = x_grad[MSB] ? (~x_grad + 1) : x_grad;  // absolute value
assign abs_y_grad = y_grad[MSB] ? (~y_grad + 1) : y_grad;  //  (~x+1 = two's-complement negate)
assign sum_xy_grad = abs_x_grad + abs_y_grad;
```

👆 The "is it negative?" test is just the top (sign) bit — and negating is `~x + 1` (two's complement, the universal hardware trick).

Finally the output must fit in **8 bits (0–255)**. A strong edge can overflow that, so it's **saturated** — clamped to 255 (`sobel_core.sv:33`):

```systemverilog
assign out_sobel_core_o = (|sum_xy_grad[hi:8]) ? MAX_PIXEL_VAL-1   // any high bit set → clamp to 255
                                               : sum_xy_grad[7:0]; // else pass through
```

▎ Result meaning: **white (255) = strong edge, black (0) = flat region.** That's the outline image you'll see on screen.


Feeding the window — `sobel_control.sv` (the FSM)

`sobel_core` is pure combinational math, but it needs **9 pixels arranged as a 3×3 window** before it can compute. Pixels arrive **one at a time** over the stream (one per `px_rdy_i` pulse). Collecting them is a job for a **state machine** — exactly the Class 2 idea, now applied to data instead of instructions (`sobel_control.sv:28`):

```systemverilog
typedef enum logic [2:0] { IDLE, FIRST_MATRIX, NEXT_MATRIX, END_FRAME } state_t;
```

- **IDLE** — wait for `start_sobel_i` (the ui[5] pin from Class 0's pinout).
- **FIRST_MATRIX** — fill the window cell by cell, steered by `counter_sobel` (`:84`):

```systemverilog
case (counter_sobel)
  0: sobel_pixels.vector0.pix0 <= in_px_sobel_i;  // top-left
  1: sobel_pixels.vector0.pix1 <= in_px_sobel_i;  // top-middle
  2: sobel_pixels.vector0.pix2 <= in_px_sobel_i;  // top-right
  3: sobel_pixels.vector1.pix0 <= in_px_sobel_i;  // ... and so on for all 9
  ...
```

- **NEXT_MATRIX** — slide the window to the next pixel and keep streaming results out, each with its own `px_rdy_o` pulse.

👆 So the control block *gathers* (sequential, clocked), the core *computes* (combinational, instant). That split — **control vs datapath** — is the single most important pattern in digital design, and you've now seen it in both the CPU (Class 2) and the filter (here).


---
🧠 Recap

1. An **edge** = a sharp change in brightness = a large **gradient**.
2. **Step 1 grayscale** collapses RGB→brightness using a **shift-add** luma approximation (`0.281R+0.562G+0.094B`) — no multiplier.
3. **Step 2 Sobel** slides two 3×3 kernels (**Gx**, **Gy**) over the window; coefficients are only 0/±1/±2, so it's all subtract-and-shift.
4. Edge strength ≈ **|Gx| + |Gy|** (cheap stand-in for √), then **saturated** to 0–255. White = edge.
5. `sobel_core` is **combinational** (the math); `sobel_control` is a **sequential FSM** that fills the 3×3 window from the pixel stream. **Control vs datapath** — the same split as the CPU.

Quick check (answer in your head 🙂)

- Why convert to grayscale *before* Sobel instead of running Sobel on R, G, B separately?
- The kernels contain only 0, ±1, ±2. Why is that a *deliberate* hardware choice, not a coincidence?
- `sobel_core` has no clock; `sobel_control` does. Which one is the datapath and which is the control — and why does that split matter?

*(Answers: brightness is what edges live in — one channel instead of three means ⅓ the logic and no colour artefacts. The coefficients are powers of two / unit values so every "multiply" becomes a shift or add — no multiplier cells on the die. `sobel_core` = datapath (combinational compute), `sobel_control` = control (sequenced gathering); separating them keeps timing clean and the math reusable.)*

---
*Next up — **Class 4: connecting CPU ↔ filter**, where we drop this whole block onto the femto bus at address `0x0045` (Option A) and let software drive it.*


📘 Class 4 — Connecting CPU ↔ filter (the merge)

This is the class where your two repos finally become **one chip**. Class 2 gave you the CPU and its bus; Class 3 gave you Diana's filter. Now we drop the filter onto the bus so **software can drive it** — this is **Option A** from the integration analysis (cell 1), made concrete.

The gap to bridge

```
   FemtoRV32  ──(memory bus: addr/data/rd/wr)──►  ??? ──►  top_gray_sobel
   speaks "addresses"                                       speaks "px_rdy + pixels"
```

The CPU only knows how to read/write **addresses**. Diana's filter only knows the **px_rdy / pixel** streaming handshake (Class 1). Something must translate between them. That something is a **peripheral wrapper** — and you already own the perfect template: `perip_mult.v`.

▎ One sentence: we wrap `top_gray_sobel` in a small block that makes it *look like memory*, so the CPU can use it with plain pointer writes.


1. What a peripheral really is (recall `perip_mult.v`)

Every femto peripheral honours the same 7-wire **contract** (`perip_mult.v:1`):

```
┌────────┬─────┬──────────────────────────────────────────────┐
│  Wire  │ Dir │                  Meaning                       │
├────────┼─────┼────────────────────────────────────────────────┤
│ cs     │ in  │ chip-select: "this message is for YOU"        │
│ addr   │ in  │ which internal register (low address bits)    │
│ wr/rd  │ in  │ is the CPU writing or reading?                │
│ d_in   │ in  │ data from CPU (on a write)                    │
│ d_out  │ out │ data back to CPU (on a read)                  │
│ clk/reset│ in│ the usual heartbeat + reset                   │
└────────┴─────┴────────────────────────────────────────────────┘
```

Inside, a peripheral does just three things (look at `perip_mult.v` — it's the whole pattern):

1. **Address-decode** `addr` into internal registers (`:20`).
2. On `cs && wr`, **capture** `d_in` into the right register (`:38`).
3. On `cs && rd`, **mux** the requested register onto `d_out` (`:56`).

The multiplier maps: `0x04→A`, `0x08→B`, `0x0C→init`, `0x10→result`, `0x14→done`. We'll do the same for Sobel.


2. Designing the Sobel register map

Diana's `top_gray_sobel` interface (from `top_gray_sobel.sv:7`) needs:

```
  in:  select_i[1:0], start_sobel_i, px_rdy_i, in_pixel_i[23:0]
  out: out_pixel_o[23:0], px_rdy_o
```

We expose each as a memory address (mirroring the mult's layout):

```
┌────────┬──────────────┬─────┬───────────────────────────────────────────┐
│ Offset │   Register   │ R/W │                 Meaning                     │
├────────┼──────────────┼─────┼─────────────────────────────────────────────┤
│ 0x04   │ PIXEL_IN     │  W  │ a 24-bit RGB pixel; writing it also pulses  │
│        │              │     │ px_rdy_i for one clock  (the handshake!)    │
│ 0x08   │ CTRL         │  W  │ bits[1:0]=select_i (mode), bit[2]=start     │
│ 0x0C   │ PIXEL_OUT    │  R  │ the processed (edge) pixel                  │
│ 0x10   │ STATUS       │  R  │ bit0 = px_rdy_o (result valid)              │
└────────┴──────────────┴─────┴───────────────────────────────────────────┘
```

⚠️ One real difference from the multiplier: the mult used a **16-bit** `d_in`, but a pixel is **24-bit** — so this peripheral uses the full **32-bit** data path. A small design decision, but exactly the kind your thesis should document.

▎ The clever bit: writing `PIXEL_IN` doesn't just store the pixel — it **generates the `px_rdy_i` pulse** Diana's pipeline expects. That single line is where "software" turns into "the Class 1 handshake."


3. The wrapper — `peripheral_sobel.v`  *(new file to add)*

This is the bridge, built by copying `perip_mult.v`'s skeleton and instantiating `top_gray_sobel` instead of `mult`:

```verilog
module peripheral_sobel(
   input         clk, reset,
   input  [31:0] d_in,
   input         cs,
   input  [4:0]  addr,
   input         rd, wr,
   output reg [31:0] d_out
);
   reg [23:0] pixel_in;   // 0x04
   reg [1:0]  mode;       // 0x08 select_i
   reg        start;      // 0x08 start_sobel
   reg        px_we;      // one-cycle px_rdy pulse

   wire [23:0] pixel_out;
   wire        px_rdy_out;

   // --- write path + handshake pulse ---
   always @(posedge clk) begin
      if (reset) begin
         pixel_in <= 0; mode <= 0; start <= 0; px_we <= 0;
      end else begin
         px_we <= 1'b0;                       // default low → pulse lasts 1 clk
         if (cs && wr) case (addr)
            5'h04: begin pixel_in <= d_in[23:0]; px_we <= 1'b1; end // load + pulse
            5'h08: begin mode <= d_in[1:0]; start <= d_in[2]; end   // control
         endcase
      end
   end

   // --- read path (mux) ---
   always @(*) begin
      d_out = 32'b0;
      if (cs && rd) case (addr)
         5'h0C: d_out = {8'b0,  pixel_out};    // processed pixel
         5'h10: d_out = {31'b0, px_rdy_out};   // status
      endcase
   end

   // --- Diana's block, unmodified ---
   top_gray_sobel filter (
      .clk_i(clk), .nreset_i(~reset),         // NOTE: filter reset is active-LOW (Class 1!)
      .select_i(mode), .start_sobel_i(start),
      .px_rdy_i(px_we), .in_pixel_i(pixel_in),
      .out_pixel_o(pixel_out), .px_rdy_o(px_rdy_out)
   );
endmodule
```

👆 Two ideas from earlier classes show up as single lines: `px_we` is the **ready/valid pulse** (Class 1); `~reset` adapts femto's active-**high** reset to Diana's active-**low** `nreset_i` (Class 1's "the `n` means active-low"). Mixing Verilog + SystemVerilog is fine — Yosys/OpenLane read both.


4. Plugging it into the SoC + driving it from C

**(a) Three edits in `SOC_flash.v`** — the same three the multiplier already has:

```verilog
// 1) widen the chip-select (7 → 8 peripherals) and decode 0x0045:
case (mem_address[31:16])
   ...
   16'h0044: cs = 8'b00000010;  // bin2bcd
   16'h0045: cs = 8'b10000000;  // sobel   ◄── NEW
endcase

// 2) instantiate the wrapper (mirror the mult instance at :100):
peripheral_sobel sobel1 (
   .clk(clk), .reset(reset), .d_in(mem_wdata),
   .cs(cs[7]), .addr(mem_address[4:0]),
   .rd(rd), .wr(wr), .d_out(sobel_dout)
);

// 3) add it to the read mux (mirror :145):
8'b10000000: mem_rdata = sobel_dout;   // ◄── NEW
```

**(b) The driver in C** — now the filter is *just memory*:

```c
#define SOBEL_PIXEL  (*(volatile int*)0x00450004)
#define SOBEL_CTRL   (*(volatile int*)0x00450008)
#define SOBEL_OUT    (*(volatile int*)0x0045000C)
#define SOBEL_STATUS (*(volatile int*)0x00450010)

SOBEL_CTRL = (0b00 << 0) | (1 << 2);   // mode 00 = full pipeline, start = 1
for (int i = 0; i < N; i++) {
   SOBEL_PIXEL = image[i];             // push RGB pixel  → pulses px_rdy
   while (!(SOBEL_STATUS & 1)) ;       // wait until result is valid
   out[i] = SOBEL_OUT;                 // read the edge pixel back
}
```

▎ Read that loop next to Class 3's pixel stream: the software `for`-loop **is** the stream, one pixel per iteration, the `while` is the handshake. Hardware and software meet here.


---
🧠 Recap

1. The CPU speaks **addresses**; the filter speaks the **px_rdy handshake**. A **peripheral wrapper** translates between them — that's the whole job.
2. Every femto peripheral obeys one 7-wire contract (`cs / addr / rd / wr / d_in / d_out / clk`). Decode → capture-on-write → mux-on-read. `perip_mult.v` is the template.
3. We map Sobel to four addresses (`PIXEL_IN / CTRL / PIXEL_OUT / STATUS`) — and writing `PIXEL_IN` **generates the `px_rdy` pulse**.
4. `peripheral_sobel.v` wraps Diana's **unmodified** `top_gray_sobel`, adapting reset polarity (`~reset`) and data width (24→32).
5. Three small `SOC_flash.v` edits (decode `0x0045`, instantiate, read-mux) + a C driver = software running the edge detector. **The two repos are now one chip.**

Quick check (answer in your head 🙂)

- Why must writing `PIXEL_IN` pulse `px_rdy` for *exactly one* clock, not hold it high?
- Diana's `top_gray_sobel` is dropped in **unmodified**. Why is that a sign of good design — and why does it matter for your thesis?
- The CPU reset is active-high but the filter's `nreset_i` is active-low. What one character fixes that, and where?

*(Answers: `top_gray_sobel` treats each `px_rdy_i` pulse as "one new pixel" — holding it high would inject the same pixel many times. Unmodified reuse means the block is already verified — you inherit Diana's testing and only have to verify the thin wrapper; that separation is a clean, defensible thesis contribution. The `~` in `.nreset_i(~reset)`, inside `peripheral_sobel.v`.)*

---
*Next up — **Class 5: making it real**, taking this merged RTL two ways: the iCESugar i9+ FPGA (prototype, live camera) and sky130/OpenLane → a Tiny Tapeout ASIC.*


📘 Class 5 — Making it real (FPGA vs ASIC)

So far everything is **RTL** — Verilog/SystemVerilog text. Class 5 turns that text into something physical. The big idea from Class 0 returns:

▎ The *same* merged RTL can go two ways — and your thesis does **both**.

```
                         ┌──────────────────────────────────────────┐
   your RTL  ───────────▶│  🔵 FPGA  (iCESugar i9+)                  │  prototype — minutes
  (femto +              │     yosys → nextpnr → bitstream            │  reprogrammable, live camera
   peripheral_sobel +    └──────────────────────────────────────────┘
   top_gray_sobel)       ┌──────────────────────────────────────────┐
              ───────────▶│  🟢 ASIC  (sky130 + OpenLane → TinyTapeout)│  product — weeks/months
                          │     yosys → OpenLane → GDSII → fab        │  permanent silicon
                          └──────────────────────────────────────────┘
```

- **FPGA** = a chip full of reconfigurable logic you *program*. Wrong? Reflash in seconds. Perfect for catching bugs and proving the camera→filter→screen path works for real.
- **ASIC** = you *manufacture* the logic into fixed silicon. No do-overs, but it's the real, tiny, low-power chip — the actual deliverable on the TinyTapeout shuttle.

Same source, two back-ends. Prove it on the FPGA first; tape it out second.


1. The FPGA path — iCESugar i9+ (prototype)

The FPGA flow is short and fast (the bottom two layers from Class 0, done in seconds):

```
  RTL ──yosys──▶ gates ──nextpnr──▶ placed&routed ──icepack──▶ bitstream ──▶ flash the board
       (synthesis)        (fit into THIS fabric)     (.bin)
```

- **yosys** turns your RTL into the FPGA's building blocks (look-up tables + flip-flops).
- **nextpnr** places & routes them into the iCE40 fabric and checks timing.
- The result is a **bitstream** you load onto the iCESugar — and your CPU+filter is *running hardware* in under a minute.

Why prototype here first? Because the FPGA can do something simulation can't: connect to the **real world** — a live camera in, a screen out, through the board's **PMOD** connectors. That's where the next piece comes in.


2. The eyes — the OV7670 camera (live image source)

To feed *real* images into your filter on the FPGA, you wire an **OV7670** camera module to the PMODs. (The datasheet is the OmniVision OV7670/OV7171 CSP2 sensor; the common product is the bare module — no FIFO — built around it.) The specs that affect your wiring:

```
┌────────────────┬────────────────────────────────────┬──────────────────────────────────────┐
│      Spec      │               Value                │              Implication               │
├────────────────┼────────────────────────────────────┼────────────────────────────────────────┤
│ I/O voltage    │ 1.7–3.0 V                          │ ⚠️ 3.3 V is slightly above spec        │
│ Digital core   │ 1.8 V                              │ from the module's onboard regulator    │
│ Analog         │ 2.45–3.0 V                         │ from the module's onboard regulator    │
│ XCLK           │ input clock (10–48 MHz typical)    │ the FPGA must generate it              │
│ Output formats │ RGB565/555/444, YUV422, raw RGB    │ RGB565 is easiest for display          │
│ Frame rate     │ up to 30 fps @ VGA 640×480         │ pixel rate ≈ 12 MHz → fast but easy    │
└────────────────┴────────────────────────────────────┴──────────────────────────────────────┘
```

Module pinout (standard bare-module variant, 18 pins):

```
┌──────────────────────────────────────────────────┐
│   1  3V3 VCC      ┃   2  GND                       │
│   3  SCL (SIOC)   ┃   4  SDA (SIOD)   ← config bus │
│   5  VS  (VSYNC)  ┃   6  HS (HREF)    ← frame sync │
│   7  PCLK (out)   ┃   8  XCLK (in, from FPGA)      │
│   9  D7           ┃  10  D6                         │
│  11  D5           ┃  12  D4           ← 8-bit pixel │
│  13  D3           ┃  14  D2             data bus    │
│  15  D1           ┃  16  D0                         │
│  17  RESET (n)    ┃  18  PWDN                       │
└──────────────────────────────────────────────────┘
```

✅ Verify against your module's silkscreen before powering up — some boards swap pin order.

How it connects to the course: the camera streams RGB565 pixels synchronized by **VSYNC/HREF/PCLK** — that is a hardware **ready/valid handshake** (Class 1) coming from outside the chip. A small capture block turns it into the 24-bit `in_pixel_i` + `px_rdy` stream that `peripheral_sobel` (Class 4) already understands. Camera → CPU → Sobel → screen, end to end.


3. The ASIC path — sky130 + OpenLane → TinyTapeout

This is the long flow: RTL all the way down to **GDSII** (the layout file the foundry manufactures). OpenLane runs the stages for you:

```
  RTL ─▶ synth ─▶ floorplan ─▶ placement ─▶ CTS ─▶ routing ─▶ signoff ─▶ GDSII
        (yosys)   (where things go)  (clock tree)         (DRC/LVS/STA)
```

Two project files steer it (from Diana's repo):

- **`config.tcl`** — `CLOCK_PERIOD = 20` ns → a **50 MHz** timing target, and `PL_TARGET_DENSITY = 0.8` (how tightly cells are packed). These are the knobs you bump if you get setup/hold violations.
- **The sky130 timing libs** you have in `OpenLane/src/sky130/` — `*_fast.lib`, `*_slow.lib`, `*_typical.lib`. These are the **process corners**: silicon comes out fast, slow, or nominal, so static timing analysis must pass at *all three*. (Slow corner → setup; fast corner → hold.)

The TinyTapeout wrapper — your chip's pins

On real silicon you don't get arbitrary ports; every TT design exposes the **same fixed pin interface** (`tt_um_gray_sobel.sv:7`):

```systemverilog
module tt_um_gray_sobel (
   input  wire [7:0] ui_in,    // 8 dedicated inputs
   output wire [7:0] uo_out,   // 8 dedicated outputs
   input  wire [7:0] uio_in,   // 8 bidirectional — input path
   output wire [7:0] uio_out,  //                 — output path
   output wire [7:0] uio_oe,   //                 — direction (1=drive)
   input  wire       ena, clk, rst_n   // enable, clock, active-low reset
);
```

Inside, it just *maps* those generic pins to your signals (`:31`) — exactly the pinout from Class 0: `ui_in[0]=spi_sck`, `ui_in[5]=start_sobel`, `uo_out[0]=spi_sdo`, etc. `uio_oe = 0` here means "all bidirectional pins used as inputs."


4. FPGA vs ASIC — same RTL, different trade-offs

```
┌────────────────┬───────────────────────────┬────────────────────────────────┐
│                │   🔵 FPGA (iCESugar i9+)  │   🟢 ASIC (sky130 / TinyTapeout)│
├────────────────┼───────────────────────────┼─────────────────────────────────┤
│ Tools          │ yosys + nextpnr           │ yosys + OpenLane               │
│ Output         │ bitstream (.bin)          │ GDSII → fabricated chip        │
│ Turnaround     │ seconds                   │ weeks–months (shuttle)         │
│ Fixable?       │ reflash anytime           │ permanent                      │
│ Real I/O       │ camera + screen via PMOD  │ fixed ui/uo/uio pins           │
│ Role           │ PROTOTYPE / debug         │ FINAL PRODUCT                  │
└────────────────┴───────────────────────────┴─────────────────────────────────┘
```

▎ The thesis story: *"I proved the camera→CPU→Sobel→screen pipeline live on the iCESugar i9+, then hardened the same RTL through OpenLane/sky130 into a TinyTapeout-ready GDSII."* Two destinations, one design — that's the strength of an RTL-based methodology.

One caution worth stating in your write-up: timing closure is **corner-dependent** on the ASIC (you must pass slow *and* fast libs), whereas on the FPGA nextpnr targets one known fabric. A design that runs at 50 MHz on the FPGA still has to *re-prove* timing at the sky130 corners.


---
🧠 Recap

1. **Same RTL, two back-ends.** Prototype on the FPGA, then tape out the identical source as an ASIC.
2. **FPGA (iCESugar i9+):** `yosys → nextpnr → bitstream`, seconds, reprogrammable, and — crucially — connects to the **real world** (camera in, screen out via PMOD).
3. **OV7670 camera:** I/O ~1.8–3.0 V, FPGA supplies **XCLK**, streams RGB565 synced by **VSYNC/HREF/PCLK** — an external ready/valid handshake feeding your `px_rdy` stream.
4. **ASIC (sky130 + OpenLane → TinyTapeout):** `RTL → … → GDSII`; `config.tcl` sets the **50 MHz** target & density; timing must pass at **slow/typical/fast** corners.
5. **TT pin interface** (`tt_um_…`): every chip exposes the same `ui_in / uo_out / uio_*` pins; your wrapper just maps them to SPI/select/start.

Quick check (answer in your head 🙂)

- Why prove the design on the FPGA *before* taping it out, when both come from the same RTL?
- Why must ASIC timing pass at **both** the slow and the fast `.lib` corner — what does each one catch?
- The OV7670's VSYNC/HREF/PCLK lines do the same job as which signal you met in Class 1?

*(Answers: an FPGA bug costs a reflash; an ASIC bug costs a whole fabrication run — the FPGA de-risks the silicon for free. Slow corner catches **setup** violations (logic too slow to make the clock edge); fast corner catches **hold** violations (data races through too quickly). They are an external **ready/valid handshake** — "this pixel/line/frame is valid now.")*

---
*Next up — **Class 6: verification & bring-up** — cocotb testbenches, gate-level vs RTL simulation, and demoing the real chip.*


---
# 🧪 Lab — El filtro Sobel en Python (modelo *dorado* del hardware)

Este laboratorio implementa en Python el **mismo filtro Sobel** que tu chip ejecuta en hardware (Clase 3, `sobel_core.sv` de Diana). Sirve como **golden model**: la referencia de software contra la cual validar el RTL.

> 📌 **Crédito / recordatorio.** Tesis de maestría (UNal) propuesta por el **estudiante de Diana Natali Maldonado Ramírez**, construida sobre su diseño TT06 `tt_um_gray_sobel`. Las imágenes de prueba reales (`flower`, `monarch`, `butterfly`) son las de su testbench `cocotb`.

**Dos aclaraciones importantes:**
1. **Sobel no se *entrena*.** No hay *training/evaluation* como en una red neuronal: los pesos del kernel son fijos (`0, ±1, ±2`). Las "1000 muestras" son un *dataset* para demostrar el flujo y validar el filtro, no para aprender nada.
2. **Kernels correctos.** Usamos `Gx=[[-1,0,1],[-2,0,2],[-1,0,1]]` y `Gy=[[-1,-2,-1],[0,0,0],[1,2,1]]` — los de tu Clase 3 y del hardware. (La versión sin signos negativos sólo desenfoca, no detecta bordes.)

**Pasos:** ① dataset 1000×(160×120) gris → ② mostrar una muestra → ③ limpieza/pre-proceso → ④ Sobel `Gx,Gy` + `|Gx|+|Gy|` (*antes → después*) → ⑤ 6–8 máscaras direccionales (compass).

In [1]:
# === Lab Sobel -- configuracion (env conda 'base') ===
import os, warnings
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color, io, transform, util, filters
from scipy.ndimage import convolve
warnings.filterwarnings("ignore")

H, W = 120, 160                      # 160x120 px  ->  filas=120, columnas=160 (gris)
RNG  = np.random.default_rng(42)     # semilla fija -> dataset reproducible
DIANA_TEST = "/Users/vic/UN/Tesis/Repository/tt06_grayscale_sobel/test"
print("conda OK -- imagen objetivo:", W, "x", H, "(WxH) en escala de grises")

conda OK -- imagen objetivo: 160 x 120 (WxH) en escala de grises


## ① Dataset — 1000 muestras de 160×120 en escala de grises

Fuentes reales: las 3 imágenes de Diana (`tt06_grayscale_sobel/test/`) + muestras de `skimage` (`coins`, `astronaut`, `camera`…). Como son ~13 imágenes fuente, generamos las **1000 muestras** con **recortes aleatorios** de 160×120 (data augmentation), con semilla fija para que sea reproducible.

In [2]:
# === Paso 1: dataset de 1000 muestras 160x120 en escala de grises ===
def load_gray(src):
    img = io.imread(src) if isinstance(src, str) else src
    if img.ndim == 3 and img.shape[2] >= 3:
        img = color.rgb2gray(img[..., :3])
    return util.img_as_float(img).astype(np.float32)

# 1a) fuentes reales de Diana (su testbench cocotb)
diana_files = ["flower_RGB.jpg", "monarch_RGB.jpg", "butterfly_640x480.jpg"]
sources, labels = [], []
for f in diana_files:
    p = os.path.join(DIANA_TEST, f)
    if os.path.exists(p):
        sources.append(load_gray(p)); labels.append(f.split("_")[0].split(".")[0])

# 1b) muestras de skimage (flower/monarch/butterfly + coins/astronaut/etc.)
ski = {"camera": data.camera, "coins": data.coins, "astronaut": data.astronaut,
       "checkerboard": data.checkerboard, "brick": data.brick, "text": data.text,
       "moon": data.moon, "horse": data.horse, "page": data.page, "rocket": data.rocket}
for name, fn in ski.items():
    try:
        sources.append(load_gray(fn())); labels.append(name)
    except Exception as e:
        print("omito", name, e)
print(f"{len(sources)} imagenes fuente:", labels)

# 1c) 1000 muestras por recortes aleatorios 160x120 (data augmentation, semilla fija)
N = 1000
X          = np.empty((N, H, W), dtype=np.float32)
sample_src = np.empty(N, dtype=int)
for i in range(N):
    j = int(RNG.integers(len(sources)))
    s = sources[j]
    if s.shape[0] < H or s.shape[1] < W:                       # ampliar si es chica
        s = transform.resize(s, (max(s.shape[0], H), max(s.shape[1], W)),
                             anti_aliasing=True).astype(np.float32)
    r = int(RNG.integers(0, s.shape[0] - H + 1))
    c = int(RNG.integers(0, s.shape[1] - W + 1))
    X[i] = s[r:r+H, c:c+W]
    sample_src[i] = j
print("dataset X:", X.shape, X.dtype, "rango", round(float(X.min()), 2), "..", round(float(X.max()), 2))

# NOTA: Sobel NO se entrena (el kernel es fijo). Si quieres usar la terminologia
# train/eval, es solo una particion del dataset -- no hay aprendizaje de pesos.
X_train, X_eval = X[:800], X[800:]
print(f"(opcional) train={len(X_train)}  eval={len(X_eval)}  -- pero el kernel es fijo, no aprende")

13 imagenes fuente: ['flower', 'monarch', 'butterfly', 'camera', 'coins', 'astronaut', 'checkerboard', 'brick', 'text', 'moon', 'horse', 'page', 'rocket']
dataset X: (1000, 120, 160) float32 rango 0.0 .. 1.0
(opcional) train=800  eval=200  -- pero el kernel es fijo, no aprende


## ② Mostrar una muestra del dataset (`plt`)

In [3]:
# === Paso 2: mostrar UNA muestra del dataset con matplotlib (plt) ===
idx = 0
plt.figure(figsize=(4, 3))
plt.imshow(X[idx], cmap="gray", vmin=0, vmax=1)
plt.title(f"Muestra #{idx}  (fuente: {labels[sample_src[idx]]})  {W}x{H} gris")
plt.axis("off"); plt.tight_layout(); plt.show()

## ③ Pre-proceso / limpieza de la imagen

Antes de Sobel se limpia el ruido (desenfoque gaussiano suave) y se cuantiza a **8 bits (0–255)** — el mismo rango del pixel en el hardware.

In [4]:
# === Paso 3: pre-proceso / limpieza de la imagen ===
def to_uint8(im):
    """normaliza a 0..255 (como el pixel de 8 bits del hardware)"""
    im = (im - im.min()) / (np.ptp(im) + 1e-9)
    return (im * 255.0).astype(np.uint8)

def preprocess(im, sigma=1.0):
    """limpieza: desenfoque gaussiano suave (quita ruido antes de Sobel)
    y cuantizacion a 8 bits (0..255)."""
    clean = filters.gaussian(im, sigma=sigma)
    return to_uint8(clean)

raw   = to_uint8(X[idx])
clean = preprocess(X[idx], sigma=1.0)

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].imshow(raw,   cmap="gray", vmin=0, vmax=255); ax[0].set_title("cruda (8 bits)");      ax[0].axis("off")
ax[1].imshow(clean, cmap="gray", vmin=0, vmax=255); ax[1].set_title("limpia (gauss s=1)");  ax[1].axis("off")
plt.tight_layout(); plt.show()

## ④ Sobel clásico: `Gx`, `Gy` y la magnitud `|Gx|+|Gy|`  — *antes → después*

`Gx` resalta bordes verticales (cambio horizontal) y `Gy` los horizontales. La fuerza del borde es la magnitud; el hardware usa la aproximación barata **`|Gx|+|Gy|`** (en vez de √Gx²+Gy²) y **satura** a 255 — replicamos exactamente eso. Cada fila va de **antes (original)** a **después (Sobel)** sobre las 3 imágenes reales de Diana.

In [5]:
# === Paso 4: Sobel clasico Gx, Gy y magnitud |Gx|+|Gy|  (ANTES -> DESPUES) ===
# Kernels CORRECTOS de Sobel (los de tu Clase 3 y del hardware sobel_core.sv).
# Lo que escribiste -- Gx=(1,0,1)/(1,0,1)/(1,0,1), Gy=(2,0,2)/(2,1,2)/(2,0,2) --
# NO detecta bordes (sin signos negativos solo suma/desenfoca). Los reales son:
Gx = np.array([[-1, 0, 1],
               [-2, 0, 2],
               [-1, 0, 1]], dtype=float)   # cambio horizontal -> bordes verticales
Gy = np.array([[-1,-2,-1],
               [ 0, 0, 0],
               [ 1, 2, 1]], dtype=float)   # cambio vertical   -> bordes horizontales

def sobel(im_u8):
    f  = im_u8.astype(float)
    gx = convolve(f, Gx, mode="reflect")
    gy = convolve(f, Gy, mode="reflect")
    mag = np.abs(gx) + np.abs(gy)                     # = |Gx|+|Gy|  (igual que el chip)
    mag_sat = np.clip(mag, 0, 255).astype(np.uint8)   # saturacion a 8 bits (hardware)
    return gx, gy, mag_sat

# aplicar a las 3 imagenes reales de Diana (completas)
real = {labels[k]: sources[k] for k in range(3)}
titles = ["ANTES (original)", "|Gx| horizontal", "|Gy| vertical", "DESPUES Sobel |Gx|+|Gy|"]
fig, ax = plt.subplots(len(real), 4, figsize=(12, 3*len(real)))
for row, (name, img) in enumerate(real.items()):
    u8 = preprocess(img, sigma=1.0)
    gx, gy, mag = sobel(u8)
    panels = [(u8, "gray", {}), (np.abs(gx), "magma", {}),
              (np.abs(gy), "magma", {}), (mag, "gray", dict(vmin=0, vmax=255))]
    for col, (d, cm, kw) in enumerate(panels):
        a = ax[row, col]; a.imshow(d, cmap=cm, **kw); a.axis("off")
        a.set_title(f"{name}: {titles[col]}" if col in (0, 3) else titles[col])
plt.suptitle("Sobel = modelo dorado del hardware de Diana  (|Gx|+|Gy|, saturado a 255)")
plt.tight_layout(); plt.show()

## ⑤ Las 6–8 máscaras Sobel direccionales (compass / brújula)

Rotando el kernel de Sobel en pasos de 45° se obtienen **8 máscaras**, cada una sensible a bordes de una dirección (N, NE, E, …, NW). Es la **detección de bordes tipo *compass*** (parecida a los operadores de Kirsch). Las **6** que citaste (N→SW, 0°–270°) son un subconjunto; aquí mostramos las 8 para comparar.

In [6]:
# === Paso 5: las 6-8 mascaras Sobel direccionales (compass / brujula) ===
# Rotando el kernel de Sobel en pasos de 45 grados se obtienen 8 mascaras,
# cada una sensible a bordes de una direccion. 6 (N..SW) u 8 es lo comun.
compass = {
 "N  (0)"  : [[ 1, 2, 1],[ 0, 0, 0],[-1,-2,-1]],
 "NE (45)" : [[ 2, 1, 0],[ 1, 0,-1],[ 0,-1,-2]],
 "E  (90)" : [[ 1, 0,-1],[ 2, 0,-2],[ 1, 0,-1]],
 "SE (135)": [[ 0,-1,-2],[ 1, 0,-1],[ 2, 1, 0]],
 "S  (180)": [[-1,-2,-1],[ 0, 0, 0],[ 1, 2, 1]],
 "SW (225)": [[-2,-1, 0],[-1, 0, 1],[ 0, 1, 2]],
 "W  (270)": [[-1, 0, 1],[-2, 0, 2],[-1, 0, 1]],
 "NW (315)": [[ 0, 1, 2],[-1, 0, 1],[-2,-1, 0]],
}
img_u8 = preprocess(sources[0], sigma=1.0)   # 'flower' de Diana

fig, ax = plt.subplots(2, 4, figsize=(13, 6))
for a, (name, k) in zip(ax.ravel(), compass.items()):
    edge = convolve(img_u8.astype(float), np.array(k, float), mode="reflect")
    a.imshow(np.abs(edge), cmap="inferno"); a.set_title(name); a.axis("off")
plt.suptitle(f"8 mascaras Sobel direccionales sobre '{labels[0]}' - bordes por direccion")
plt.tight_layout(); plt.show()
print("Las 6 primeras (N,NE,E,SE,S,SW = 0-270) son el conjunto que citaste; +W,NW completan 8.")

Las 6 primeras (N,NE,E,SE,S,SW = 0-270) son el conjunto que citaste; +W,NW completan 8.


### 🏋️ Ejercicio — las 8 máscaras compass aplicadas a las 3 imágenes = **24 imágenes**

Cada fila es una imagen real de Diana (`flower`, `monarch`, `butterfly`) y cada columna una dirección (N, NE, E, SE, S, SW, W, NW). En total **8 × 3 = 24** imágenes filtradas, para comparar cómo cada dirección resalta bordes distintos en cada imagen.

In [7]:
# === Ejercicio 1 (COMPASS): 8 mascaras direccionales x 3 imagenes = 24 (3 filas x 8) ===
# Pantalla ancha: figura grande para que NO se vean encogidas.
ejercicio = {labels[k]: sources[k] for k in range(3)}   # flower, monarch, butterfly
filas, cols = len(ejercicio), len(compass)              # 3 x 8
fig, ax = plt.subplots(filas, cols, figsize=(3.0*cols, 3.4*filas))   # ~24 x 10 (widescreen)
total = 0
for r, (nombre, img) in enumerate(ejercicio.items()):
    base = preprocess(img, sigma=1.0).astype(float)
    for c, (dirn, k) in enumerate(compass.items()):
        edge = convolve(base, np.array(k, float), mode="reflect")
        a = ax[r, c]
        a.imshow(np.abs(edge), cmap="inferno")
        if r == 0: a.set_title(dirn, fontsize=12)
        if c == 0:
            a.set_ylabel(nombre, fontsize=13, rotation=90); a.set_xticks([]); a.set_yticks([])
        else:
            a.axis("off")
        total += 1
plt.suptitle(f"COMPASS: {cols} direcciones x {filas} imagenes = {total} imagenes", fontsize=15)
plt.tight_layout(); plt.show()
print("total compass:", total, "(esperado 24)")

total compass: 24 (esperado 24)


### 🏋️ Ejercicio 2 (DIRECCIONAL) — `Gx`, `Gy` y `√(Gx²+Gy²)` sobre 8 imágenes = **24 imágenes**

3 **filas** = las tres operaciones (`Gx` horizontal, `Gy` vertical, magnitud real `√(Gx²+Gy²)`) y 8 **columnas** = 8 imágenes (flower, monarch, butterfly + 5 de skimage). En total **3 × 8 = 24**.

> Nota: aquí uso la magnitud **real** `√(Gx²+Gy²)` (Pitágoras). El hardware usa la aproximación barata `|Gx|+|Gy|` (Paso 4) — compáralas.

In [8]:
# === Ejercicio 2 (DIRECCIONAL): Gx, Gy, sqrt(Gx^2+Gy^2) sobre 8 imagenes = 24 (3 filas x 8) ===
imgs8, names8 = sources[:8], labels[:8]          # 3 de Diana + 5 de skimage
G   = [preprocess(im, sigma=1.0).astype(float) for im in imgs8]
GXs = [convolve(g, Gx, mode="reflect") for g in G]
GYs = [convolve(g, Gy, mode="reflect") for g in G]
MAG = [np.sqrt(gx**2 + gy**2) for gx, gy in zip(GXs, GYs)]   # magnitud REAL (Pitagoras)

rows = [("Gx (horizontal)",       GXs, "magma"),
        ("Gy (vertical)",         GYs, "magma"),
        ("sqrt(Gx^2 + Gy^2)",     MAG, "gray")]
fig, ax = plt.subplots(3, 8, figsize=(3.0*8, 3.4*3))    # widescreen ~24 x 10
total = 0
for r, (etiqueta, data_row, cmap) in enumerate(rows):
    for c in range(8):
        a = ax[r, c]
        a.imshow(np.abs(data_row[c]), cmap=cmap)
        if r == 0: a.set_title(names8[c], fontsize=11)
        if c == 0:
            a.set_ylabel(etiqueta, fontsize=12); a.set_xticks([]); a.set_yticks([])
        else:
            a.axis("off")
        total += 1
plt.suptitle(f"DIRECCIONAL: Gx / Gy / sqrt(Gx^2+Gy^2) sobre {len(imgs8)} imagenes = {total} imagenes", fontsize=15)
plt.tight_layout(); plt.show()
print("total direccional:", total, "(esperado 24)")

total direccional: 24 (esperado 24)


---
## 🧠 Recap y conexión con tu hardware

- El **dataset** (1000×160×120 gris) demuestra el flujo; **Sobel no aprende** — el kernel es fijo (`0, ±1, ±2`).
- `|Gx|+|Gy|` saturado a 8 bits aquí en Python = exactamente lo que hace `sobel_core.sv` (Clase 3). Este notebook es tu **golden model**: pasa la misma imagen por el RTL (cocotb) y compara contra `mag_sat` — deben coincidir pixel a pixel.
- Las **máscaras compass** son una extensión: además de "¿hay borde?" responden "¿en qué dirección?".

**Siguiente paso sugerido:** exportar una muestra 160×120 a un archivo de pixeles (hex) para alimentar el testbench `cocotb` de Diana (`tt06_grayscale_sobel/test/`) y verificar que el hardware reproduce el `|Gx|+|Gy|` de Python.

---
## ⑥ Exportar una muestra 160×120 a `.hex` / `.txt` para el testbench `cocotb` de Diana

El testbench de Diana (`gray_sobel_TB.py`) consume **un pixel por línea, row-major**: escribe el valor con `f.write(f"{int(str(pixel),2)}\n")` → **decimal** por línea (0–255 en gris). Exportamos la misma muestra en dos formatos:

- **`.hex`** — 2 dígitos hex por línea (formato `$readmemh` de Verilog, lo que pediste).
- **`.txt`** — decimal por línea (el formato **exacto** que ya lee su testbench).

Además exportamos el **Sobel golden** de software (`|Gx|+|Gy|` saturado): alimentas el `input` al RTL en cocotb y comparas su salida contra este golden — deben coincidir pixel a pixel.

In [9]:
# === Paso 6: exportar una muestra 160x120 a .hex / .txt para el testbench cocotb de Diana ===
OUT_DIR = "export"   # cambia a la ruta del test de Diana para alimentarlo directo:
                     # "/Users/vic/UN/Tesis/Repository/tt06_grayscale_sobel/test"
os.makedirs(OUT_DIR, exist_ok=True)

sample_in = clean                          # 120x160 uint8 (0..255), muestra ya limpia
_, _, sample_sobel = sobel(sample_in)      # |Gx|+|Gy| saturado -> Sobel golden de software

def save_pixels(path_base, arr_u8):
    """escribe arr (HxW uint8) fila por fila (row-major):
       .hex -> 2 digitos hex por linea ($readmemh) ; .txt -> decimal por linea (formato Diana)."""
    flat = arr_u8.reshape(-1)
    with open(path_base + ".hex", "w") as fh, open(path_base + ".txt", "w") as ft:
        for px in flat:
            fh.write(f"{int(px):02X}\n")
            ft.write(f"{int(px)}\n")
    return len(flat)

n_in  = save_pixels(os.path.join(OUT_DIR, "sample_160x120_input"),        sample_in)
n_out = save_pixels(os.path.join(OUT_DIR, "sample_160x120_sobel_golden"), sample_sobel)

print(f"escrito en {os.path.abspath(OUT_DIR)}/")
print(f"  sample_160x120_input.hex/.txt         ({n_in} pixeles = {W}x{H}, row-major, gris 8 bits)")
print(f"  sample_160x120_sobel_golden.hex/.txt  ({n_out} pixeles, |Gx|+|Gy| saturado a 255)")
print("primeras 5 lineas .hex:", [f"{int(px):02X}" for px in sample_in.reshape(-1)[:5]])
# Uso en cocotb: por cada linea de *_input.txt -> int(linea) -> dut.in_pixel_i;
# captura out_pixel_o y comparalo contra *_sobel_golden (deben coincidir pixel a pixel).

escrito en /Users/vic/UN/Tesis/Repository/tty_project_filter/conda/TTY_Filter_Sobel/export/
  sample_160x120_input.hex/.txt         (19200 pixeles = 160x120, row-major, gris 8 bits)
  sample_160x120_sobel_golden.hex/.txt  (19200 pixeles, |Gx|+|Gy| saturado a 255)
primeras 5 lineas .hex: ['10', '30', '51', '48', '22']
